# Phase 2G - Does the pseudo-patient recovery recover patients?

**Run on:** Kaggle or Colab. **No GPU needed.** ~20-30 min, mostly download.

---

### The gap this closes

The leakage measurement rests on a grouping recovered from image similarity,
because IQ-OTH/NCCD distributes no patient identifiers. Two things about that
grouping are known and uncomfortable:

1. The similarity threshold was chosen by minimising the distance between the
   recovered component count and the documented 110 cases. It is fitted to a
   target, not tested against one.
2. The class-wise recovery disagrees with the documented composition in two
   directions at once: benign is over-split (29 groups vs 15 cases) and normal
   is under-recovered (45 vs 55), so the error is correlated with class label.

Neither can be resolved on IQ-OTH/NCCD, because there is nothing to check
against. So this notebook checks against a dataset that *does* carry identity.

### The design

**MSD Task06_Lung** ships one volume per case. Sample axial slices from each
volume and the true case label of every slice is known by construction. Then:

| step | what |
|---|---|
| 1 | Sample ~10 slices per case, imitating IQ-OTH/NCCD's ~10 slices per case |
| 2 | Run the **identical** Phase 0 procedure: 64x64 grey thumbnail, mean-centred, L2-normalised, cosine similarity, threshold, connected components |
| 3 | Score recovered clusters against true case labels: ARI, V-measure, and pairwise precision / recall / F1 |
| 4 | Repeat across a threshold sweep |
| 5 | Apply the paper's own selection rule (component count nearest the known case count) and report the quality **at that threshold** |

Step 5 is the one that matters. It reproduces exactly what we did on
IQ-OTH/NCCD, with the answer available, so it estimates what that grouping is
actually worth.

### What would falsify our approach

If, at the fitted threshold, pairwise precision is high but recall is low, the
procedure over-splits: groups are pure but fragmented, residual leakage remains,
and our reported effect is conservative. If **precision** is low, the procedure
merges distinct patients, the grouped partition is not patient-level in any
sense, and the paper's central interpretation does not hold. We report whichever
we find.

### One caveat, stated up front

MSD volumes are HU-valued CT reconstructed by us into 8-bit slices, while
IQ-OTH/NCCD ships images already rendered by someone else. Image formation
therefore differs, and the fidelity measured here is an estimate for a
comparable collection rather than a certificate for that specific one.

## 1. Environment

In [ ]:
import subprocess, sys
for pkg in ["nibabel", "opencv-python-headless", "tabulate", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
                   check=False)

import os, json, time, shutil, random
import numpy as np
import pandas as pd
import nibabel as nib
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (adjusted_rand_score, v_measure_score,
                             homogeneity_score, completeness_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK    = "/kaggle/working" if IN_KAGGLE else "/content"
SCRATCH = "/kaggle/temp"    if IN_KAGGLE else "/content"
try:
    os.makedirs(SCRATCH, exist_ok=True)
except OSError:
    SCRATCH = "/tmp"; os.makedirs(SCRATCH, exist_ok=True)

RESULTS_DIR = f"{WORK}/fyp_phase2g_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = {"seed": SEED}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)

print("environment:", "Kaggle" if IN_KAGGLE else "Colab", "| outputs ->", RESULTS_DIR)

## 2. MSD Task06_Lung

Only `imagesTr` is extracted. The segmentation labels are irrelevant here: the
ground truth we need is *which case a slice came from*, and that is the filename.

In [ ]:
MSD_URL = "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar"
ROOT = f"{SCRATCH}/msd"
os.makedirs(ROOT, exist_ok=True)

IMG_DIR = f"{ROOT}/Task06_Lung/imagesTr"
if not os.path.isdir(IMG_DIR):
    cmd = (f"curl -L --retry 3 --fail '{MSD_URL}' | "
           f"tar -x -C {ROOT} --wildcards --exclude='._*' 'Task06_Lung/imagesTr/*'")
    t0 = time.time()
    subprocess.run(cmd, shell=True, check=True)
    print(f"downloaded + extracted in {(time.time()-t0)/60:.1f} min")

vols = sorted(f for f in os.listdir(IMG_DIR)
              if f.endswith(".nii.gz") and not f.startswith("."))
print(f"{len(vols)} case volumes")
RESULTS["n_cases"] = len(vols)

## 3. Build a collection with the same shape as IQ-OTH/NCCD

IQ-OTH/NCCD is ~1097 images from ~110 cases, so roughly ten slices per case. We
sample the same way: ten evenly spaced slices from the central 60% of each
volume, where lung is present. Slices are lung-windowed and written as 8-bit
greyscale, which is the form the grouping procedure expects.

`SLICES_PER_CASE` is fixed in advance, not tuned.

In [ ]:
SLICES_PER_CASE = 10
LUNG_WINDOW = (-1000.0, 400.0)      # standard lung window in Hounsfield units

def to_uint8(sl):
    lo, hi = LUNG_WINDOW
    sl = np.clip(sl, lo, hi)
    return ((sl - lo) / (hi - lo) * 255.0).astype(np.uint8)

rows = []
for vf in tqdm(vols, desc="slicing"):
    vol = nib.load(os.path.join(IMG_DIR, vf))
    arr = vol.get_fdata(dtype=np.float32)          # (H, W, Z)
    z = arr.shape[2]
    lo, hi = int(0.20 * z), int(0.80 * z)
    if hi - lo < SLICES_PER_CASE:
        lo, hi = 0, z
    idx = np.linspace(lo, hi - 1, SLICES_PER_CASE).astype(int)
    case = vf.replace(".nii.gz", "")
    for k in idx:
        img = to_uint8(arr[:, :, k].T)             # (H, W), radiological view
        rows.append({"case": case, "z": int(k),
                     "img": cv2.resize(img, (512, 512),
                                       interpolation=cv2.INTER_AREA)})

df = pd.DataFrame([{"case": r["case"], "z": r["z"]} for r in rows])
IMGS = [r["img"] for r in rows]
true_labels = pd.factorize(df["case"])[0]
print(f"{len(df)} slices from {df['case'].nunique()} cases "
      f"({len(df)/df['case'].nunique():.1f} per case)")
RESULTS["n_slices"] = int(len(df))
RESULTS["slices_per_case"] = SLICES_PER_CASE
save_json()

## 4. The Phase 0 grouping procedure, unchanged

Copied from Phase 0 rather than reimplemented, so that what is measured here is
the procedure the paper actually used.

In [ ]:
thumbs = []
for img in tqdm(IMGS, desc="thumbnails"):
    g = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    g -= g.mean()
    n = np.linalg.norm(g)
    thumbs.append(g / n if n > 0 else g)
T = np.stack(thumbs)
S = T @ T.T
np.fill_diagonal(S, 0.0)
print("similarity matrix:", S.shape)

def components(thresh):
    parent = list(range(len(S)))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    ii, jj = np.where(np.triu(S, 1) >= thresh)
    for a, b in zip(ii, jj):
        ra, rb = find(int(a)), find(int(b))
        if ra != rb: parent[ra] = rb
    roots = [find(i) for i in range(len(S))]
    remap = {r: k for k, r in enumerate(sorted(set(roots)))}
    return np.array([remap[r] for r in roots])

## 5. Score the recovery against true case identity

Cluster-level agreement (ARI, V-measure) plus the pairwise view, which is the
one that maps onto leakage:

- **Pairwise precision** - of the image pairs the procedure puts in one group,
  the fraction genuinely from the same case. Low precision means *merging*
  distinct patients, which breaks the patient-level interpretation.
- **Pairwise recall** - of the pairs genuinely from one case, the fraction the
  procedure catches. Low recall means *over-splitting*, which leaves residual
  leakage and makes our reported effect conservative.

In [ ]:
def pairwise(true, pred):
    # counts over pairs, computed from cluster contingency rather than O(n^2)
    df_ = pd.DataFrame({"t": true, "p": pred})
    both = sum(c * (c - 1) // 2 for c in df_.groupby(["t", "p"]).size())
    same_pred = sum(c * (c - 1) // 2 for c in df_.groupby("p").size())
    same_true = sum(c * (c - 1) // 2 for c in df_.groupby("t").size())
    prec = both / same_pred if same_pred else float("nan")
    rec = both / same_true if same_true else float("nan")
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else float("nan")
    return prec, rec, f1

THRS = [0.80, 0.85, 0.90, 0.925, 0.95, 0.96, 0.97, 0.98, 0.99]
sweep = []
for t in THRS:
    pred = components(t)
    prec, rec, f1 = pairwise(true_labels, pred)
    sweep.append({"threshold": t,
                  "n_groups": int(pred.max() + 1),
                  "ari": float(adjusted_rand_score(true_labels, pred)),
                  "v_measure": float(v_measure_score(true_labels, pred)),
                  "homogeneity": float(homogeneity_score(true_labels, pred)),
                  "completeness": float(completeness_score(true_labels, pred)),
                  "pair_precision": float(prec),
                  "pair_recall": float(rec),
                  "pair_f1": float(f1)})

sw = pd.DataFrame(sweep)
print(sw.to_string(index=False))
sw.to_csv(f"{RESULTS_DIR}/grouping_validation_sweep.csv", index=False)
RESULTS["sweep"] = sweep
save_json()

## 6. The test that matters: quality at the *fitted* threshold

Phase 0 picked its threshold by minimising the distance between the recovered
component count and the known case count. Here we apply that same rule, with the
true grouping available but unused by the rule itself, and then look at how good
the resulting clustering actually is.

This is the closest available estimate of what the IQ-OTH/NCCD grouping is worth.

In [ ]:
KNOWN = df["case"].nunique()
best = min(sweep, key=lambda r: abs(r["n_groups"] - KNOWN))
RESULTS["known_cases"] = int(KNOWN)
RESULTS["fitted_threshold"] = best

print(f"true cases                 : {KNOWN}")
print(f"threshold picked by the rule: {best['threshold']}  "
      f"-> {best['n_groups']} groups")
print()
print(f"  adjusted Rand index : {best['ari']:.3f}")
print(f"  V-measure           : {best['v_measure']:.3f}")
print(f"  homogeneity         : {best['homogeneity']:.3f}   (1.0 = no merging)")
print(f"  completeness        : {best['completeness']:.3f}   (1.0 = no splitting)")
print(f"  pairwise precision  : {best['pair_precision']:.3f}")
print(f"  pairwise recall     : {best['pair_recall']:.3f}")
print()
if best["pair_precision"] >= 0.90:
    print("PRECISION IS HIGH: groups are mostly pure, so the grouped partition is")
    print("close to patient-level. Any error is over-splitting, which leaves residual")
    print("leakage and makes the reported effect CONSERVATIVE.")
else:
    print("PRECISION IS LOW: the procedure merges images from different cases, so the")
    print("grouped partition is NOT patient-level and the paper's interpretation of")
    print("the effect does not follow. This must be reported.")
save_json()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(sw["threshold"], sw["n_groups"], "o-", color="#2471a3")
axes[0].axhline(KNOWN, color="grey", ls=":", label=f"true cases ({KNOWN})")
axes[0].set_xlabel("cosine threshold"); axes[0].set_ylabel("recovered groups")
axes[0].set_title("Component count"); axes[0].legend(); axes[0].grid(alpha=.3)

for col, lab in [("pair_precision", "pairwise precision"),
                 ("pair_recall", "pairwise recall"),
                 ("ari", "adjusted Rand index")]:
    axes[1].plot(sw["threshold"], sw[col], "o-", label=lab)
axes[1].axvline(best["threshold"], color="crimson", ls="--",
                label=f"fitted threshold ({best['threshold']})")
axes[1].set_xlabel("cosine threshold"); axes[1].set_ylim(0, 1.02)
axes[1].set_title("Recovery quality vs ground truth")
axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_grouping_validation.png", dpi=200)
plt.show()

## 7. Bundle

In [ ]:
path = shutil.make_archive(f"{WORK}/fyp_phase2g_results", "zip", RESULTS_DIR)
print("archive:", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
if not IN_KAGGLE:
    try:
        from google.colab import files; files.download(path)
    except Exception as e:
        print("download from the file browser:", e)
else:
    print("Kaggle: under /kaggle/working, use the Output panel")
print()
print("Report BOTH the sweep and the fitted-threshold row in the paper, whichever")
print("way they come out. The point of this notebook is that the answer was not")
print("chosen in advance.")